In [1]:
# Mount Google Drive
from google.colab import drive
#drive.mount('/content/drive',force_remount=True)
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
RAW = "/content/drive/MyDrive/tll/raw_data"

df_items = pd.read_csv(f"{RAW}/items.csv")
df_stores = pd.read_csv(f"{RAW}/stores.csv")
df_events = pd.read_csv(f"{RAW}/holidays_events.csv")
dataset = pd.read_csv("/content/drive/MyDrive/tll_reproducibility/intermediate_files/weekly_corrected_dataset.csv")
dataset.date = pd.to_datetime(dataset.date)
dataset.shape

(11293583, 11)

In [ ]:
# df_items = pd.read_csv("/content/drive/MyDrive/tll_reproducibility/items.csv")
# df_stores = pd.read_csv("/content/drive/MyDrive/tll_reproducibility/intermediate_files/stores.csv")
# df_events = pd.read_csv("../../data/favorita_rawdata/holidays_events.csv")
# dataset = pd.read_csv("../../data/intermediante_files/weekly_corrected_dataset.csv")
# dataset.date = pd.to_datetime(dataset.date)
# dataset.shape

(11293583, 11)

In [4]:
dataset.head()

,date,store_nbr,item_nbr,unit_sales,unit_sales_corrected,onpromotion,family,class,perishable,type,cluster
0,2014-04-06,1.0,105574.0,5.0,7.700000,0.0,GROCERY I,1045.0,0.0,D,13.0
1,2014-04-06,1.0,105575.0,66.0,66.000000,0.0,GROCERY I,1045.0,0.0,D,13.0
2,2014-04-06,1.0,106716.0,16.0,16.000000,0.0,GROCERY I,1032.0,0.0,D,13.0
3,2014-04-06,1.0,108696.0,17.0,18.454545,0.0,DELI,2636.0,1.0,D,13.0
4,2014-04-06,1.0,108786.0,9.0,10.000000,0.0,CLEANING,3044.0,0.0,D,13.0


In [5]:
dataset_features = dataset.copy()

In [6]:
dataset_features.unit_sales_corrected = dataset_features.unit_sales_corrected.astype(np.float32)
dataset_features.unit_sales = dataset_features.unit_sales.astype(np.float32)
dataset_features.store_nbr = dataset_features.store_nbr.astype(np.int32)
dataset_features.item_nbr = dataset_features.item_nbr.astype(np.int32)
dataset_features.onpromotion = dataset_features.onpromotion.astype(bool)

In [7]:
dataset_features.memory_usage()

,0
Index,132
date,90348664
store_nbr,45174332
item_nbr,45174332
unit_sales,45174332
unit_sales_corrected,45174332
onpromotion,11293583
family,90348664
class,90348664
perishable,90348664


# Step 0 - remove negatives

In [8]:
dataset_features.unit_sales_corrected = dataset_features.unit_sales_corrected.apply(lambda x: max(0, x))

# Step 1 - log sales and log price

In [9]:
dataset_features.unit_sales = dataset_features.unit_sales.apply(np.log1p)
dataset_features.unit_sales_corrected = dataset_features.unit_sales_corrected.apply(np.log1p)
print("Converted to log")

Converted to log


# Step 2 - calculate rolling stats

In [10]:
def add_rolling_stats(df, sales_col, windows=[]):

    df = df.set_index("date")

    for window in windows:
        df["rolling_avg_{}_w_{}".format(sales_col, window)]    = df[sales_col].shift(1)\
                                                                  .rolling(window=window, min_periods=0, center=False)\
                                                                  .mean()
        df["rolling_median_{}_w_{}".format(sales_col, window)] = df[sales_col].shift(1)\
                                                                  .rolling(window=window, min_periods=0, center=False)\
                                                                  .median()
        df["rolling_std_{}_w_{}".format(sales_col, window)]    = df[sales_col].shift(1)\
                                                                  .rolling(window=window, min_periods=0, center=False)\
                                                                  .std()

    df = df.reset_index()

    return df

dataset_features = dataset_features\
                     .groupby(['store_nbr', 'item_nbr'], as_index=False)\
                     .apply(lambda df: add_rolling_stats(df, "unit_sales_corrected", [4, 12])).reset_index()
dataset_features = dataset_features.drop(['level_0', 'level_1'], axis=1)
print("Calculated rolling stats")

/tmp/ipykernel_1835/4166077938.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: add_rolling_stats(df, "unit_sales_corrected", [4, 12])).reset_index()


Calculated rolling stats


In [11]:
dataset_features.columns

Index(['date', 'store_nbr', 'item_nbr', 'unit_sales', 'unit_sales_corrected',
       'onpromotion', 'family', 'class', 'perishable', 'type', 'cluster',
       'rolling_avg_unit_sales_corrected_w_4',
       'rolling_median_unit_sales_corrected_w_4',
       'rolling_std_unit_sales_corrected_w_4',
       'rolling_avg_unit_sales_corrected_w_12',
       'rolling_median_unit_sales_corrected_w_12',
       'rolling_std_unit_sales_corrected_w_12'],
      dtype='object')

# Step 3 - build lag matrix

In [12]:
dataset_lagged_features = []
for store_nbr in dataset_features.store_nbr.unique():
    print("processing store {}".format(store_nbr), end="\r")
    tmp = dataset_features.loc[dataset_features.store_nbr == store_nbr]\
                [["date", "store_nbr", "item_nbr", "unit_sales_corrected", "onpromotion",
                  'rolling_avg_unit_sales_corrected_w_4','rolling_median_unit_sales_corrected_w_4','rolling_std_unit_sales_corrected_w_4',
                  'rolling_avg_unit_sales_corrected_w_12','rolling_median_unit_sales_corrected_w_12','rolling_std_unit_sales_corrected_w_12']]\
                .groupby(["date", "item_nbr", "store_nbr"]).first()\
                .unstack("item_nbr")\
                .reset_index()

    lagged_store_dataset_list = []
    # sales lags
    for lag in [1,2,3,4,52]:
        tmp2 = tmp[["unit_sales_corrected"]].shift(lag).add_suffix('_lag_{}'.format(lag))
        lagged_store_dataset_list.append(tmp2)
    # promotion lags
    for lag in [0,1,2,3,4]:
        tmp2 = tmp[["onpromotion"]].shift(lag).add_suffix('_lag_{}'.format(lag))
        lagged_store_dataset_list.append(tmp2)
    # rolling stats
    lagged_store_dataset_list.append(tmp[['rolling_avg_unit_sales_corrected_w_4',
                                          'rolling_median_unit_sales_corrected_w_4',
                                          'rolling_std_unit_sales_corrected_w_4',
                                          'rolling_avg_unit_sales_corrected_w_12',
                                          'rolling_median_unit_sales_corrected_w_12',
                                          'rolling_std_unit_sales_corrected_w_12']])

    # prepend target
    lagged_store_dataset_list.insert(0, tmp[["unit_sales_corrected"]])

    # date and store
    lagged_store_dataset_list.insert(0, tmp[["date", "store_nbr"]])

    dataset_store_lagged_features = pd.concat(lagged_store_dataset_list, axis=1)

    dataset_lagged_features.append(dataset_store_lagged_features)

# make final dataset
dataset_lagged_features = pd.concat(dataset_lagged_features, axis=0)

print("Lagged features created", end="\r")

# Step 4 - build categorical matrix

In [13]:
dataframe_product_categories = pd.get_dummies(df_items.set_index("item_nbr"),
                                              columns=['family', "perishable"])\
                                 .reset_index()\
                                 .drop(["class"], axis=1)

dataframe_store_categories   = pd.get_dummies(df_stores.set_index("store_nbr"),
                                              columns=['type', "cluster"])\
                                 .reset_index()\
                                 .drop(["state", "city"], axis=1)

print("Categorical features created", end="\r")

# Step 5 - build calendar events

In [14]:
unique_dates = pd.to_datetime(dataset_features.date.unique())

df_events.date = pd.to_datetime(df_events.date)
df_events.description = df_events.description.apply(lambda x: x.split('-')[0].split("+")[0])
dataset_calendar_features = df_events[df_events.locale=="National"]\
                                .set_index("date")\
                                .groupby([pd.Grouper(freq="W"), "description"])\
                                .count().unstack("description").fillna(0)["type"]

dataset_calendar_features = pd.concat([dataset_calendar_features, dataset_calendar_features.shift(1).add_suffix("_lag_1")], axis=1)

dataset_calendar_features = dataset_calendar_features.reindex(unique_dates).fillna(0)

print("Calendar features created", end="\r")

# Step 6 - build seasonal variables

In [15]:
unique_dates = pd.to_datetime(dataset_features.date.unique())

first_date = min(unique_dates)

trend         = [(d-first_date).days for d in unique_dates]

monthly_cycle_sin = [np.sin(2*np.pi*t/4) for t in trend]
monthly_cycle_cos = [np.cos(2*np.pi*t/4) for t in trend]

yearly_cycle_sin = [np.sin(2*np.pi*t/52) for t in trend]
yearly_cycle_cos = [np.cos(2*np.pi*t/52) for t in trend]

data_dict = {
    "trend": trend,
    "monthly_cycle_sin": monthly_cycle_sin,
    "monthly_cycle_cod": monthly_cycle_cos,
    "yearly_cycle_sin": yearly_cycle_sin,
    "yearly_cycle_cos": yearly_cycle_cos
}

dataframe_seasonal_features = pd.DataFrame(data_dict)

print("Seasonal features created", end="\r")

# Step 7 - export feature files

In [16]:
# calendar
dataset_calendar_features.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/calendar_features.csv", index=False)
# categories
dataframe_product_categories.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/product_features.csv", index=False)
dataframe_store_categories.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/store_features.csv", index=False)
# seasonal
dataframe_seasonal_features.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/seasonal_features.csv", index=False)
# lagged
dataset_lagged_features.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/lagged_features_2.csv", index=False)

/tmp/ipykernel_1835/3209372145.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset_lagged_features.fillna(0).to_csv("/content/drive/MyDrive/tll_reproducibility/calculated_features/lagged_features_2.csv", index=False)
